# Analise Exploratoria de Dados (EDA)
## Pipeline de Risco de Credito

**Autora:** Nayane Araujo  
**GitHub:** [Nayanearaujo](https://github.com/Nayanearaujo)  
**Projeto:** credit-risk-data-pipeline  

---

### O que vamos fazer neste notebook?

Este e o primeiro notebook do projeto. Aqui a gente vai explorar os dados de credito para entender o que temos antes de limpar e modelar qualquer coisa.

A ideia e simples: **nao modelamos o que nao entendemos.** Antes de treinar qualquer modelo de Machine Learning, precisamos responder perguntas como:

- Qual e o perfil dos clientes que tomam credito?
- Quem e mais propenso a dar calote (inadimplencia)?
- Existem padroes claros nos dados que um modelo poderia aprender?
- Ha dados faltando ou inconsistentes?

### Dataset utilizado

O dataset e o **Kaggle Credit Risk Dataset**, com 32.581 registros de emprestimos pessoais. Cada linha representa um pedido de credito com informacoes como:

- Dados do tomador: idade, renda, tipo de moradia, tempo de emprego
- Dados do emprestimo: valor, taxa de juros, finalidade, grade de risco
- Historico de credito: anos de historico, inadimplencia previa
- **Target:** `loan_status` (0 = pagou em dia, 1 = nao pagou)

### Estrutura do notebook

1. Importacoes e configuracao
2. Carregamento dos dados (camada Bronze)
3. Visao geral do dataset
4. Analise do target (inadimplencia)
5. Analise das variaveis numericas
6. Analise das variaveis categoricas
7. Correlacoes e relacoes entre variaveis
8. Valores nulos e qualidade dos dados
9. Conclusoes e proximos passos

---

## 1. Importacoes e Configuracao

Vamos comecar importando as bibliotecas que precisamos. Cada uma tem um papel especifico:

- **pandas**: manipulacao e analise de dados em tabelas (DataFrames)
- **numpy**: operacoes matematicas e arrays
- **matplotlib e seaborn**: graficos estaticos e bonitos
- **plotly**: graficos interativos (da para passar o mouse e ver os valores)
- **warnings**: para nao poluir a saida com avisos desnecessarios

In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Configuracao visual padrao para graficos matplotlib/seaborn
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

# Configuracao de exibicao do pandas (ver mais colunas de uma vez)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Bibliotecas carregadas com sucesso!')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'seaborn : {sns.__version__}')

## 2. Carregamento dos Dados (Camada Bronze)

Na Arquitetura Medallion que usamos neste projeto, os dados brutos ficam na camada **Bronze**. Eles chegam exatamente como foram coletados, sem nenhuma alteracao.

Isso e importante por dois motivos:
1. **Rastreabilidade**: sempre podemos voltar ao dado original se precisarmos
2. **Integridade**: sabemos que nao alteramos nada antes de analisar

Se o arquivo Bronze nao existir ainda, o codigo vai baixar automaticamente usando o script de ingestao.

In [ ]:
# Caminho para os dados brutos (camada Bronze)
BRONZE_PATH = Path('../data/bronze/credit_risk_raw.csv')

# Se o arquivo nao existir, vamos baixar agora
if not BRONZE_PATH.exists():
    print('Arquivo Bronze nao encontrado. Baixando agora...')
    sys.path.insert(0, str(Path('..').resolve()))
    from src.ingestion.ingest_kaggle import run_ingestion
    df_raw = run_ingestion()
else:
    df_raw = pd.read_csv(BRONZE_PATH)
    print(f'Dados carregados com sucesso: {BRONZE_PATH}')

print(f'\nShape do dataset: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas')

## 3. Visao Geral do Dataset

Antes de qualquer grafico, vamos conhecer o dataset de perto. O que temos aqui?

In [ ]:
# Primeiras 5 linhas para ver o formato dos dados
print('Primeiras linhas do dataset:')
df_raw.head()

In [ ]:
# Informacoes sobre tipos de dados e valores nulos
print('Informacoes gerais do dataset:')
df_raw.info()

In [ ]:
# Estatisticas descritivas basicas
# Isso nos da uma visao rapida de medias, desvios, minimos e maximos
print('Estatisticas descritivas das variaveis numericas:')
df_raw.describe().round(2)

In [ ]:
# Analise de valores nulos - fundamental para planejar a limpeza
nulls = df_raw.isnull().sum()
null_pct = (nulls / len(df_raw) * 100).round(2)

null_report = pd.DataFrame({
    'Coluna': nulls.index,
    'Qtd Nulos': nulls.values,
    'Percentual (%)': null_pct.values
}).query('`Qtd Nulos` > 0').sort_values('Qtd Nulos', ascending=False)

print('Colunas com valores nulos:')
if len(null_report) == 0:
    print('Nenhuma coluna com valores nulos encontrada.')
else:
    print(null_report.to_string(index=False))

# Visualizacao dos nulos
if len(null_report) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(null_report['Coluna'], null_report['Percentual (%)'], color='#e74c3c')
    ax.set_xlabel('Percentual de valores nulos (%)')
    ax.set_title('Porcentagem de Valores Nulos por Coluna', fontweight='bold')
    for bar, pct in zip(bars, null_report['Percentual (%)']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=11)
    plt.tight_layout()
    plt.savefig('../docs/eda_nulls.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Analise do Target: Inadimplencia (loan_status)

O target do nosso projeto e a coluna `loan_status`:
- **0 = Adimplente**: o cliente pagou o emprestimo em dia
- **1 = Inadimplente**: o cliente nao pagou (calote)

Antes de qualquer coisa, precisamos saber: **o dataset e balanceado?** Ou seja, temos uma proporcao razoavel de cada classe?

Isso e fundamental porque, se tivermos 95% de adimplentes e 5% de inadimplentes, um modelo que chuta sempre adimplente ja teria 95% de acuracia, mas seria inutil na pratica.

Por isso a acuracia sozinha nao e uma boa metrica para credito. Vamos usar AUC-ROC e F1-Score.

In [ ]:
# Distribuicao do target
target_counts = df_raw['loan_status'].value_counts()
target_pct = df_raw['loan_status'].value_counts(normalize=True) * 100

print('Distribuicao do target loan_status:')
print(f'  Adimplente  (0): {target_counts[0]:,} registros ({target_pct[0]:.1f}%)')
print(f'  Inadimplente(1): {target_counts[1]:,} registros ({target_pct[1]:.1f}%)')
print(f'  Razao de desbalanceamento: {target_counts[0]/target_counts[1]:.1f}:1')

# Grafico de pizza e barras lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pizza
cores = ['#2ecc71', '#e74c3c']
labels = ['Adimplente (0)', 'Inadimplente (1)']
axes[0].pie(target_counts, labels=labels, colors=cores, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Proporcao de Adimplentes x Inadimplentes', fontweight='bold')

# Barras
axes[1].bar(labels, target_counts.values, color=cores, width=0.5)
axes[1].set_title('Quantidade por Classe', fontweight='bold')
axes[1].set_ylabel('Numero de registros')
for i, (v, pct) in enumerate(zip(target_counts.values, target_pct.values)):
    axes[1].text(i, v + 200, f'{v:,}\n({pct:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/eda_target_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nObservacao: O dataset tem desbalanceamento moderado.')
print('Isso e esperado em problemas de credito: inadimplencia e rara, mas custosa.')
print('Vamos usar SMOTE na etapa de modelagem para lidar com isso.')

## 5. Analise das Variaveis Numericas

Vamos explorar cada variavel numerica e ver como ela se comporta para adimplentes e inadimplentes.

O que buscamos:
- Ha diferenca de distribuicao entre as duas classes? (Se sim, e um bom preditor)
- Ha outliers? (valores extremos que podem ser erros ou casos excepcionais)
- A distribuicao e normal ou assimetrica? (importa na hora de escalonar)

In [ ]:
# Seleciona variaveis numericas
num_cols = ['person_age', 'person_income', 'person_emp_length',
            'loan_amnt', 'loan_int_rate', 'loan_percent_income',
            'cb_person_cred_hist_length']

# Nomes mais legiveis para os graficos
col_labels = {
    'person_age': 'Idade',
    'person_income': 'Renda Anual',
    'person_emp_length': 'Anos de Emprego',
    'loan_amnt': 'Valor do Emprestimo',
    'loan_int_rate': 'Taxa de Juros (%)',
    'loan_percent_income': 'Comprometimento de Renda',
    'cb_person_cred_hist_length': 'Historico de Credito (anos)'
}

fig, axes = plt.subplots(len(num_cols), 2, figsize=(16, 4 * len(num_cols)))

for i, col in enumerate(num_cols):
    label = col_labels.get(col, col)
    
    # Histograma por classe
    for status, cor, nome in [(0, '#2ecc71', 'Adimplente'), (1, '#e74c3c', 'Inadimplente')]:
        axes[i, 0].hist(
            df_raw[df_raw['loan_status'] == status][col].dropna(),
            bins=30, alpha=0.6, color=cor, label=nome, density=True
        )
    axes[i, 0].set_title(f'Distribuicao: {label}', fontweight='bold')
    axes[i, 0].set_xlabel(label)
    axes[i, 0].set_ylabel('Densidade')
    axes[i, 0].legend()
    
    # Boxplot por classe
    data_adim = df_raw[df_raw['loan_status'] == 0][col].dropna()
    data_inad = df_raw[df_raw['loan_status'] == 1][col].dropna()
    bp = axes[i, 1].boxplot([data_adim, data_inad], patch_artist=True,
                             labels=['Adimplente', 'Inadimplente'])
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    axes[i, 1].set_title(f'Boxplot: {label}', fontweight='bold')
    axes[i, 1].set_ylabel(label)

plt.suptitle('Analise das Variaveis Numericas por Status de Inadimplencia',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../docs/eda_numeric_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comparacao estatistica: media de cada variavel por classe
print('Media das variaveis numericas por status de inadimplencia:')
comparison = df_raw.groupby('loan_status')[num_cols].mean().round(2)
comparison.index = ['Adimplente (0)', 'Inadimplente (1)']

# Calcula diferenca percentual
diff_pct = ((comparison.loc['Inadimplente (1)'] - comparison.loc['Adimplente (0)']) /
            comparison.loc['Adimplente (0)'] * 100).round(1)
comparison.loc['Diferenca (%)'] = diff_pct

print(comparison.T.to_string())

print('\nInsight principal:')
print('Inadimplentes tem taxa de juros media MAIOR, valor de emprestimo MAIOR')
print('e comprometimento de renda MAIOR. Isso faz sentido de negocio!')

## 6. Analise das Variaveis Categoricas

As variaveis categoricas descrevem caracteristicas qualitativas dos clientes e emprestimos:

- `person_home_ownership`: tipo de moradia (aluguel, propria, financiada)
- `loan_intent`: finalidade do emprestimo (pessoal, educacao, saude, negocio, casa, dividas)
- `loan_grade`: classificacao de risco do emprestimo (A = menor risco, G = maior risco)
- `cb_person_default_on_file`: ja teve inadimplencia antes? (Y/N)

Para cada variavel, vamos ver a taxa de inadimplencia por categoria. Isso mostra quais grupos sao mais arriscados.

In [ ]:
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
cat_labels = {
    'person_home_ownership': 'Tipo de Moradia',
    'loan_intent': 'Finalidade do Emprestimo',
    'loan_grade': 'Grade de Risco',
    'cb_person_default_on_file': 'Inadimplencia Previa'
}

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    label = cat_labels[col]
    
    # Taxa de inadimplencia por categoria
    taxa = df_raw.groupby(col)['loan_status'].agg(['mean', 'count']).reset_index()
    taxa.columns = ['categoria', 'taxa_inadimplencia', 'total']
    taxa['taxa_pct'] = taxa['taxa_inadimplencia'] * 100
    taxa = taxa.sort_values('taxa_pct', ascending=False)
    
    # Cor por taxa de risco
    cores = plt.cm.RdYlGn_r(taxa['taxa_pct'] / taxa['taxa_pct'].max())
    
    bars = axes[idx].bar(taxa['categoria'], taxa['taxa_pct'], color=cores)
    axes[idx].set_title(f'Taxa de Inadimplencia por {label}', fontweight='bold')
    axes[idx].set_xlabel(label)
    axes[idx].set_ylabel('Taxa de Inadimplencia (%)')
    axes[idx].set_ylim(0, taxa['taxa_pct'].max() * 1.2)
    
    # Adiciona valores nas barras
    for bar, (_, row) in zip(bars, taxa.iterrows()):
        axes[idx].text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f'{row["taxa_pct"]:.1f}%\n(n={row["total"]:,})',
            ha='center', va='bottom', fontsize=9
        )

plt.suptitle('Taxa de Inadimplencia por Variavel Categorica', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/eda_categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analise detalhada da grade de risco (uma das mais importantes)
grade_analysis = df_raw.groupby('loan_grade').agg(
    total=('loan_status', 'count'),
    inadimplentes=('loan_status', 'sum'),
    taxa_inadimplencia=('loan_status', 'mean'),
    taxa_juros_media=('loan_int_rate', 'mean'),
    valor_medio=('loan_amnt', 'mean')
).round(2)

grade_analysis['taxa_inadimplencia_pct'] = (grade_analysis['taxa_inadimplencia'] * 100).round(1)
grade_analysis = grade_analysis.drop('taxa_inadimplencia', axis=1)

print('Analise completa por Grade de Risco:')
print(grade_analysis.to_string())
print('\nInsight: A grade e um excelente preditor!')
print('A taxa de inadimplencia sobe progressivamente de A para G.')

## 7. Correlacoes e Relacoes entre Variaveis

O mapa de correlacao nos mostra o quanto cada variavel esta associada com as outras.

Valores proximos de **+1** indicam correlacao positiva forte: quando uma sobe, a outra tende a subir.  
Valores proximos de **-1** indicam correlacao negativa: quando uma sobe, a outra tende a cair.  
Valores proximos de **0** indicam que as variaveis nao tem relacao linear entre si.

Para o nosso modelo, quanto mais correlacionada com `loan_status`, mais util e a variavel.

In [ ]:
# Prepara dataset para correlacao (apenas numericas)
df_corr = df_raw[num_cols + ['loan_status']].copy()

# Mapa de correlacao
fig, ax = plt.subplots(figsize=(12, 8))
mask = np.triu(np.ones_like(df_corr.corr(), dtype=bool))
sns.heatmap(
    df_corr.corr(),
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=-1, vmax=1,
    center=0,
    ax=ax,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Mapa de Correlacao das Variaveis Numericas', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlacoes com o target
corr_target = df_corr.corr()['loan_status'].drop('loan_status').sort_values(ascending=False)
print('Correlacao de cada variavel com o target (loan_status):')
for col, val in corr_target.items():
    sinal = 'positiva' if val > 0 else 'negativa'
    print(f'  {col:35s}: {val:+.3f} ({sinal})')

In [ ]:
# Grafico interativo: taxa de juros vs comprometimento de renda
# Colorido por status de inadimplencia
df_sample = df_raw.sample(n=min(5000, len(df_raw)), random_state=42).copy()
df_sample['Status'] = df_sample['loan_status'].map({0: 'Adimplente', 1: 'Inadimplente'})

fig = px.scatter(
    df_sample,
    x='loan_int_rate',
    y='loan_percent_income',
    color='Status',
    color_discrete_map={'Adimplente': '#2ecc71', 'Inadimplente': '#e74c3c'},
    opacity=0.6,
    title='Taxa de Juros x Comprometimento de Renda (amostra de 5.000 registros)',
    labels={
        'loan_int_rate': 'Taxa de Juros (%)',
        'loan_percent_income': 'Comprometimento de Renda (%)'
    },
    hover_data=['person_age', 'person_income', 'loan_amnt', 'loan_grade']
)
fig.update_layout(height=500)
fig.show()

print('Passe o mouse sobre os pontos para ver detalhes de cada registro!')

## 8. Analise de Outliers

Outliers sao valores muito diferentes da maioria dos dados. Podem ser:
- **Erros de digitacao**: idade de 144 anos, renda negativa
- **Casos excepcionais reais**: uma pessoa muito rica ou muito velha

Precisamos identificar e decidir o que fazer com eles antes de treinar o modelo.

In [ ]:
# Detecta outliers usando o metodo IQR (Interquartile Range)
# Outlier: valor abaixo de Q1 - 1.5*IQR ou acima de Q3 + 1.5*IQR

print('Deteccao de outliers (metodo IQR):')
print(f'{"Coluna":35s} {"Q1":>10s} {"Q3":>10s} {"Outliers":>10s} {"Pct":>8s}')
print('-' * 75)

for col in num_cols:
    serie = df_raw[col].dropna()
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((serie < lower) | (serie > upper)).sum()
    pct = outliers / len(serie) * 100
    print(f'{col:35s} {Q1:10.1f} {Q3:10.1f} {outliers:10,} {pct:7.1f}%')

print('\nIdades extremas no dataset:')
print(df_raw[df_raw['person_age'] > 80][['person_age', 'person_income', 'loan_status']].head(10))
print(f'\nTotal de pessoas com mais de 80 anos: {(df_raw["person_age"] > 80).sum()}')
print('Decisao: remover idades acima de 100 anos (veja silver_transform.py)')

## 9. Conclusoes e Proximos Passos

Excelente! Fizemos uma analise exploratoria completa. Veja o que aprendemos:

In [ ]:
print('=' * 65)
print('RESUMO DA ANALISE EXPLORATORIA')
print('=' * 65)

print('\n1. DATASET')
print(f'   Total de registros : {len(df_raw):,}')
print(f'   Total de variaveis : {len(df_raw.columns)}')
print(f'   Taxa de inadimplencia: {df_raw["loan_status"].mean()*100:.1f}%')

print('\n2. QUALIDADE DOS DADOS')
print(f'   Colunas com nulos: loan_int_rate e person_emp_length')
print(f'   Estrategia: mediana por grade (taxa de juros) e mediana global (emprego)')
print(f'   Outliers: idades > 100 anos serao removidas')

print('\n3. PRINCIPAIS PREDITORES DE INADIMPLENCIA')
print('   a) loan_grade: taxa sobe de 11% (A) ate 33% (G)')
print('   b) loan_int_rate: inadimplentes pagam taxas maiores')
print('   c) loan_percent_income: quanto mais comprometida a renda, maior o risco')
print('   d) cb_person_default_on_file: historico de calote e forte preditor')
print('   e) loan_intent: finalidade importa (dividas tem risco maior)')

print('\n4. PROXIMOS PASSOS')
print('   -> Notebook 02: Feature Engineering (limpeza + codificacao + SMOTE)')
print('   -> Notebook 03: Treinamento e Avaliacao do Modelo')
print('   -> Notebook 04: Azure e Databricks (ambiente de nuvem)')
print('=' * 65)

---

### Continue a jornada

Agora que entendemos os dados, vamos para o **Notebook 02: Feature Engineering**, onde vamos preparar os dados para o modelo de Machine Learning.

**Nayane Araujo** | [GitHub](https://github.com/Nayanearaujo)